# One kernel, two memory levels
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/demos/week6/Memory_Levels_Demo_Colab.ipynb)

**Dataflow Architectures for AI, Week 6, Module 2: Bytes per memory level**

A roofline has one memory roof per memory level. The byte count in the arithmetic intensity is not one number: it depends on which level you count the bytes at. This notebook runs the same kernel on buffers that fit in the GPU's L2 cache and on buffers that do not. The FLOPs per byte are the same at every size. The achieved bandwidth is not, because the bytes come from a different memory level.

**What you will do**
1. Print the GPU and its L2 cache size.
2. Measure the DRAM roof with a read-only stream.
3. Run one streaming kernel on buffers below and above the L2 size, and plot the bandwidth.
4. Place an L2-resident point and a DRAM point on a roofline with two roofs.
5. See why a library matrix-vector product is a poor probe for a memory level.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells in order. The notebook takes about a minute, most of it compiling one Triton kernel.

## 1. Check the environment
The L2 cache size decides which buffers fit. It is 4 MB on a T4, 48 MB on an L4, 40 MB on an A100 and 50 MB on an H100.

In [ ]:
import statistics
import torch
import triton
import triton.language as tl

assert torch.cuda.is_available(), "This notebook needs a GPU: Runtime > Change runtime type > T4 GPU"
props = torch.cuda.get_device_properties(0)
L2 = props.L2_cache_size
SMS = props.multi_processor_count
print("PyTorch:", torch.__version__, "| Triton:", triton.__version__)
print(f"GPU: {props.name}, {SMS} streaming multiprocessors")
print(f"L2 cache: {L2 / 2**20:.1f} MiB")

## 2. Measure the DRAM roof
The kernel in this notebook only reads its data, so the roof is measured with a kernel that only reads: a sum over a 256 MiB buffer, far larger than any L2 cache. A copy would count a read and a write, and on most GPUs it reports less than a read-only stream reaches.

In [ ]:
big = torch.ones(128 * 2**20, dtype=torch.float16, device="cuda")      # 256 MiB
ms = triton.testing.do_bench(lambda: big.sum())
bw_dram = big.numel() * big.element_size() / (ms * 1e-3)
del big
print(f"DRAM roof (read-only stream): {bw_dram / 1e9:.0f} GB/s")

## 3. One streaming kernel at many sizes
The kernel reads a buffer of FP16 values in blocks of 2048 and adds each block to a running sum. It does one addition per value, 1 FLOP per 2 bytes, so its arithmetic intensity is 0.5 FLOPs per byte at every size.

Three details make it measure the memory level, not something else:

- **Enough work per launch.** Each launch reads about 512 MiB in total, passing over a small buffer many times. One launch then lasts long enough that launch latency does not matter.
- **A different reader each pass.** A fixed set of programs, four per streaming multiprocessor, walks the buffer. Program `p` reads blocks `p`, `p + G`, `p + 2G` and so on, where `G` is the number of programs, wrapping around the end of the buffer. On the next pass, a block is read by a different program on a different multiprocessor, so the data cannot come from that multiprocessor's own L1 cache.
- **Loads that skip L1.** `cache_modifier=".cg"` tells the GPU to cache the loads in L2 only.

The buffer sizes are chosen from this GPU's L2 size: four that fit (a quarter to 90 percent of L2) and three that do not (1.5, 4 and 16 times L2).

In [ ]:
@triton.jit
def stream_kernel(x_ptr, out_ptr, n_blocks, total_blocks, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    G = tl.num_programs(0)
    acc = tl.zeros((BLOCK,), dtype=tl.float32)
    for i in range(pid, total_blocks, G):            # pass over the buffer several times
        block = i % n_blocks                          # wrap around: a different block each step
        offsets = block * BLOCK + tl.arange(0, BLOCK)
        acc += tl.load(x_ptr + offsets, cache_modifier=".cg").to(tl.float32)   # L2 only, not L1
    tl.store(out_ptr + pid * BLOCK + tl.arange(0, BLOCK), acc)


BLOCK = 2048
PROGRAMS = 4 * SMS
READ_PER_LAUNCH = 512 * 2**20                         # bytes read by one launch


def stream_bandwidth(nbytes):
    n_blocks = max(1, nbytes // (2 * BLOCK))
    x = torch.randn(n_blocks * BLOCK, device="cuda", dtype=torch.float16)
    out = torch.empty(PROGRAMS * BLOCK, device="cuda", dtype=torch.float32)
    total = max(n_blocks, READ_PER_LAUNCH // (2 * BLOCK))
    ms = triton.testing.do_bench(lambda: stream_kernel[(PROGRAMS,)](x, out, n_blocks, total, BLOCK=BLOCK))
    return n_blocks * BLOCK * 2, total * BLOCK * 2 / (ms * 1e-3)


stream_rows = []
for f in (0.25, 0.5, 0.75, 0.9, 1.5, 4, 16):
    size, bw = stream_bandwidth(int(f * L2))
    stream_rows.append((size, bw))
    where = "fits in L2" if size < L2 else "larger than L2"
    print(f"buffer {size / 2**20:8.1f} MiB  {bw / 1e9:7.0f} GB/s  {bw / bw_dram:5.2f} x DRAM roof  ({where})")

**Reading the output**
- The largest buffers run at about 1.0 times the DRAM roof. On every pass their bytes stream from device memory.
- The buffers that fit in L2 run faster than the DRAM roof: on a T4, about two to four times faster. After the first pass, every byte they read comes from L2.
- On a T4 the 1.5 times L2 buffer sits in between: part of it stays in L2 and part of it is evicted on every pass.
- The kernel and its 0.5 FLOPs per byte are the same on every line. Only the memory level changed.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.plot([s / 2**20 for s, _ in stream_rows], [b / 1e9 for _, b in stream_rows], "o-", color="black",
        label="streaming kernel, measured")
ax.axhline(bw_dram / 1e9, color="0.4", linestyle="--", label="DRAM roof (read-only stream)")
ax.axvline(L2 / 2**20, color="0.6", linestyle=":", label="L2 size")
ax.set_xscale("log")
ax.set_xlabel("buffer size (MiB)")
ax.set_ylabel("achieved GB/s")
ax.set_ylim(bottom=0)
ax.legend(frameon=False, fontsize=8)
ax.set_title(f"Same kernel, same FLOPs per byte ({props.name})")
plt.tight_layout()
plt.show()

## 4. Two points, two roofs
The cell places the fastest L2-resident buffer and the largest buffer on a roofline. Both do 0.5 FLOPs per byte. It draws two memory roofs: the DRAM roof from section 2, and an L2 roof taken from the best L2-resident measurement.

In [ ]:
import numpy as np

AI = 0.5                                              # FLOPs per byte for this kernel
in_l2 = max((r for r in stream_rows if r[0] < L2), key=lambda r: r[1])
largest = stream_rows[-1]
bw_l2 = in_l2[1]

ai = np.logspace(-1.5, 1.5, 200)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.loglog(ai, bw_dram * ai / 1e9, color="black", label="DRAM roof")
ax.loglog(ai, bw_l2 * ai / 1e9, color="0.55", linestyle="--", label="L2 roof (measured)")
ax.loglog([AI], [AI * in_l2[1] / 1e9], "o", markersize=9, color="0.25", label="buffer fits in L2")
ax.loglog([AI], [AI * largest[1] / 1e9], "s", markersize=9, color="0.6", label="buffer streams from DRAM")
ax.set_xlabel("arithmetic intensity (FLOPs per byte)")
ax.set_ylabel("GFLOP/s")
ax.legend(frameon=False, fontsize=8)
ax.set_title("One arithmetic intensity, two memory roofs")
plt.tight_layout()
plt.show()
print(f"L2 roof is {bw_l2 / bw_dram:.1f} times the DRAM roof on this GPU.")

**Reading the output**
- Against the DRAM roof alone, the L2-resident point is far above the roof. A point above a roof means its bytes were counted at the wrong level.
- Against the L2 roof, the same point is on its roof. Counted at the level its bytes actually came from, it is a memory-bound kernel running at that level's bandwidth.
- The largest buffer sits on the DRAM roof.
- To place a kernel correctly, count its bytes at each memory level. A kernel can be bound by one level for one size and by another level for a different size.

## 5. Why not a matrix-vector product?
A decode step is mostly matrix-vector products, so it is natural to try this experiment with `y = W x` instead of a kernel written for it. On most GPUs the result is unclear, for two reasons:

- **One call is short.** A matrix that fits in a 4 MB L2 is read in a few microseconds. That is too little work to fill every streaming multiprocessor and too short to hide the kernel's start-up latency, so the call is limited by latency before it reaches either roof. Replaying calls from a CUDA graph removes the launch cost, but not this latency.
- **The kernel changes with the shape.** cuBLAS picks a different matrix-vector kernel for different sizes, and some reach a much smaller fraction of the bandwidth than others. A change in bandwidth between two sizes can then come from the kernel choice, not from the memory level.

The streaming kernel in section 3 avoids both: every size runs the same code, and every launch reads 512 MiB. The general rule is to measure a memory level with a kernel built for the purpose, then use the result to explain a real kernel. The Week 6 decode-step demo in this folder shows one such case: on an L4, whose 48 MB L2 holds the 32 MB weight matrix, the batch-1 projection lands above its DRAM roof.

## 6. Same notebook on other GPUs
The same notebook was run on four NVIDIA GPUs: the T4 from Colab's free tier, the L4, the A100 and the H100. The lecture marks values like these as MEASURED, observed on hardware, and SNAPSHOT, valid for one software release. Read the table for the pattern across GPUs, not for exact numbers.

<!-- gpu-table:start -->
| GPU | L2, MiB | DRAM roof, GB/s | fits in L2 | 1.5x L2 | 16x L2 |
|---|---|---|---|---|---|
| T4 (cc 7.5) | 4 | 266 | 1,022 GB/s (3.8x) | 491 GB/s (1.8x) | 266 GB/s (1.0x) |
| L4 (cc 8.9) | 48 | 228 | 1,682 GB/s (7.4x) | 245 GB/s (1.1x) | 248 GB/s (1.1x) |
| A100-40GB (cc 8.0) | 40 | 1,202 | 2,291 GB/s (1.9x) | 1,415 GB/s (1.2x) | 1,352 GB/s (1.1x) |
| H100 (cc 9.0) | 50 | 2,465 | 6,122 GB/s (2.5x) | 2,531 GB/s (1.0x) | 2,612 GB/s (1.1x) |

Streaming kernel from section 3. In brackets: achieved bandwidth as a multiple of the DRAM roof. "Fits in L2" is the fastest size below the L2 size.

*MEASURED on cloud GPUs on 2026-11-18, PyTorch 2.15.0. SNAPSHOT: these values change with library versions and with the host CPU, and your Colab T4 can differ from the T4 row.*
<!-- gpu-table:end -->

**What the comparison shows**
- On every GPU, the streaming kernel runs faster than the DRAM roof while its buffer fits in L2, and drops to the DRAM roof once it does not.
- The ratio between the two levels differs by GPU. It is largest on the L4, whose GDDR6 memory has modest bandwidth, and smallest on the A100 and H100, whose HBM is already fast. L2 helps most where DRAM is slowest.
- The L2 size decides where the drop happens: about 4 MB on the T4 and 40 to 50 MB on the others.
- On a scratchpad machine such as Spyre there is no cache to make this decision. The compiler's placement plan decides which level each tensor lives in, which is the subject of the rest of this module.

## Try this
1. In section 3, remove `cache_modifier=".cg"`. Do the small buffers get faster? Which cache is serving them now?
2. In section 3, change `READ_PER_LAUNCH` to the buffer size, so each launch reads the buffer once. What happens to the L2-resident sizes, and why?
3. Time `y = W x` for a matrix that fits in L2 with `triton.testing.do_bench`, and compare its bandwidth with the streaming kernel at the same size. Use section 5 to explain the difference.

## Summary
- The same kernel with the same FLOPs per byte runs at very different bandwidths, depending on which memory level its bytes come from.
- A point above the DRAM roof means the bytes came from a faster level. Count them at that level and compare with that level's roof.
- Measuring a memory level needs a kernel built for it, with enough work per launch. A short library kernel is limited by latency and by its own efficiency.
- On a scratchpad machine, the compiler's placement plan makes the same decision explicitly.

Lecture: Week 6, Module 2, slides 18 and 19 (one arithmetic intensity per level, bandwidth set by software). Reference: Williams, Waterman and Patterson, "Roofline", CACM 2009.